# Machine Learning-Based Decoding for Planar Surface Codes

- **Author**: Théo HUET
- **Date**: 21/10/2026

In [1]:
!pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc

This notebook builds directly upon our previous work, ["Introduction to the Surface Code"](https://github.com/TheoHUETQC/surface-code/blob/main/surface_code_introduction.ipynb), where we implemented a distance-3 ($d=3$) planar surface code using Qiskit and a classical Look-up Table decoder.

While a Look-up Table (LUT) is effective for single-qubit errors ($weight = 1$), it suffers from two major limitations:
1. **Exponential Scaling:** As the code distance $d$ increases, the number of possible syndromes grows exponentially, making static lookup tables unfeasible.
2. **Multiple Errors (Weight $> 1$):** LUTs struggle to handle overlapping or higher-weight errors without drastically expanding in complexity.

So we're going to use some code from this notebook without explaining it, it allows us to obtain the syndromes and the quantum circuit.

In [2]:
# from surface_code_introduction.ipynb
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

d = 3
nbr_logical_qubits = 1
nbr_data_qubits = d * d
nbr_ancilla_qubits = nbr_data_qubits - nbr_logical_qubits
nbr_ancilla_x = nbr_ancilla_qubits // 2
nbr_ancilla_z = nbr_ancilla_qubits - nbr_ancilla_x

def init_registers():
  q_data = QuantumRegister(nbr_data_qubits, name="data")
  q_anc_z = QuantumRegister(nbr_ancilla_z, name="ancilla_z")
  q_anc_x = QuantumRegister(nbr_ancilla_x, name="ancilla_x")
  c_syndrome_z = ClassicalRegister(nbr_ancilla_z, name="syndrome_z")
  c_syndrome_x = ClassicalRegister(nbr_ancilla_x, name="syndrome_x")
  return q_data, q_anc_z, q_anc_x, c_syndrome_z, c_syndrome_x

def stabilizer_circuit(qc, q_data, q_anc_z, q_anc_x, c_syndrome_z, c_syndrome_x):
  map_Z = {0: [0,1,3,4], 1: [4,5,7,8], 2: [1,2], 3: [6,7]}
  qc.barrier(label="Z-Stabilizer")
  for anc_Z_idx in range(nbr_ancilla_z):
    for data_idx in map_Z[anc_Z_idx]:
      qc.cx(q_data[data_idx], q_anc_z[anc_Z_idx])

  map_X = {0: [1,2,4,5], 1: [3,4,6,7], 2: [0,3], 3: [5,8]}
  qc.barrier(label="X-Stabilizer")
  for anc_X_idx in range(nbr_ancilla_x):
    qc.h(q_anc_x[anc_X_idx])
    for data_idx in map_X[anc_X_idx]:
      qc.cx(q_anc_x[anc_X_idx], q_data[data_idx])
    qc.h(q_anc_x[anc_X_idx])

  qc.barrier(label="Measurement")
  qc.measure(q_anc_z, c_syndrome_z)
  qc.measure(q_anc_x, c_syndrome_x)
  return qc

def build_surface_code_circuit(q_data, errors_to_inject=[], init_basis='Z'):
  q_anc_z, q_anc_x, c_syndrome_z, c_syndrome_x = init_registers()[1:]
  qc = QuantumCircuit(q_data, q_anc_z, q_anc_x, c_syndrome_z, c_syndrome_x, name="stabilizer_circuit")

  if init_basis == 'X':
    qc.barrier(label="State Prep")
    qc.h(q_data)

  qc.barrier(label="Error Injection")
  for error in errors_to_inject:
    if error[0] == 'X':
      qc.x(q_data[error[1]])
    elif error[0] == 'Z':
      qc.z(q_data[error[1]])

  qc = stabilizer_cluster_qc = stabilizer_circuit(qc, q_data, q_anc_z, q_anc_x, c_syndrome_z, c_syndrome_x)
  return qc

def simulate_surface_code(q_data, errors_to_inject=[], shots=100, init_basis='Z'):
  from qiskit_aer import AerSimulator
  from qiskit import transpile
  qc = build_surface_code_circuit(q_data, errors_to_inject, init_basis)
  simulator = AerSimulator()
  compiled_circuit = transpile(qc, simulator)
  job = simulator.run(compiled_circuit, shots=shots)
  counts = job.result().get_counts()
  return counts, qc

def recover_error_related_syndrome(q_data, errors_to_inject, shots=100):
  counts_Z, qc = simulate_surface_code(q_data, errors_to_inject, shots, init_basis='Z')
  most_common_z = max(counts_Z, key=counts_Z.get).split(" ")[1]
  counts_X, _ = simulate_surface_code(q_data, errors_to_inject, shots, init_basis='X')
  most_common_x = max(counts_X, key=counts_X.get).split(" ")[0]
  syndrome_x_str = most_common_x
  syndrome_z_str = most_common_z
  return syndrome_x_str, syndrome_z_str

### The Machine Learning Solution
To overcome these limitations, this project implements a **Neural Network Decoder using PyTorch**. Instead of hardcoding a static table, we will train a Multi-Layer Perceptron (MLP) to learn the non-linear mapping between error syndromes $(S_X, S_Z)$ and the optimal physical corrections.

### Objectives of this Notebook:
1. **Dataset Generation:** Automatically generate synthetic error-syndrome pairs using our Qiskit surface code simulation pipeline.
2. **PyTorch Architecture:** Build a classification neural network tailored for quantum error correction.
3. **Training & Evaluation:** Train the model, evaluate its accuracy, and compare its flexibility against the traditional Look-up Table approach.

### Understanding Error Weight
The **weight** of a Pauli error corresponds to the number of physical qubits simultaneously affected by a non-trivial error operator ($X$, $Y$, or $Z$).
- **Weight-1 errors** affect a single qubit (e.g., `[('X', 1)]` or a single $Y$ error on qubit 1). For a distance $d=3$ surface code, weight-1 errors are fully correctable.
- **Weight-2 errors** affect two distinct qubits simultaneously (e.g., `[('X', 1), ('Z', 4)]`). For a distance $d=3$ code, the code's correction capacity ($\lfloor(d-1)/2\rfloor = 1$) is exceeded, meaning weight-2 errors typically lead to uncorrectable logical failures.

In [3]:
def generate_possible_errors(weight=1):
  def get_pauli_opts(q):
    return [
      [('X', q)],                  # X
      [('Z', q)],                  # Z
      [('Z', q), ('X', q)]         # Y
      ]

  possible_errors = []
  nbr_qubits = 9
  qubits = range(nbr_qubits)

  for q in qubits: # error with weight == 1
    for opt in get_pauli_opts(q):
      possible_errors.append(opt)

  possible_errors.append([]) # Identity error

  if weight == 1:
    return possible_errors

  elif weight == 2:
   for q1 in qubits:
      for q2 in range(q1 + 1, nbr_qubits):
        for opt1 in get_pauli_opts(q1):
          for opt2 in get_pauli_opts(q2):
            possible_errors.append(opt1 + opt2)

  else:
    raise ValueError("Weight must be 1 or 2")
  return possible_errors

---

# 1. Dataset

In [4]:
import numpy as np
import torch

def generate_dataset_from_errors(num_samples=1000, weight=1):
  X_data = [] # Will contain the syndromes as binary vectors (4+4 bits total)
  y_data = [] # Will contain the index of the corresponding error in 'possible_errors'

  q_data, _, _, _, _ = init_registers()

  possible_errors = generate_possible_errors(weight=weight)

  for i in range(num_samples):
    if i % (num_samples*0.1) == 0:
      print(f"Generating sample {i}/{num_samples}")

    err_idx = np.random.randint(0, len(possible_errors))
    error_to_inject = possible_errors[err_idx]

    s_x, s_z = recover_error_related_syndrome(q_data, errors_to_inject=error_to_inject, shots=50)

    # Convert binary strings to lists of integers (e.g., '0011' -> [0, 0, 1, 1])
    vector = [int(bit) for bit in s_x + s_z]

    X_data.append(vector)
    y_data.append(err_idx)

  return torch.tensor(X_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)
  #return np.array(X_data, dtype=np.float32), np.array(y_data, dtype=np.int64)

---

# 2. Preprocessing

`TensorDataset` is a container that pairs your input data $X$ with your labels $y$, row by row.

`DataLoader` is a PyTorch tool for managing training in mini-batches. It takes your `TensorDataset` and automatically handles the following:

- Shuffling the data at each epoch (`shuffle=True`), which is crucial to prevent the model from learning the order of the examples.
- Splitting your dataset into fixed-size batches (e.g., `batch_size=64`).

In [5]:
from torch.utils.data import TensorDataset, DataLoader

def preprocess_dataset(X_tensor, y_tensor):
  dataset = TensorDataset(X_tensor, y_tensor)
  dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
  return dataloader

In [6]:
def generate_torch_dataset(num_samples=1000, weight=1):
  X_tensor, y_tensor = generate_dataset_from_errors(num_samples=num_samples, weight=weight)
  print("Dataset Size:", X_tensor.shape)
  return preprocess_dataset(X_tensor, y_tensor)

---

# 3. model building

In [7]:
import torch.nn as nn
import torch.nn.functional as F

class SurfaceCodeDecoderMLP(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=64, output_dim=28):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

---

# 4. model training

## 4.1 optimization

In [8]:
import torch.optim as optim

def init_training(model):
  criterion = nn.CrossEntropyLoss() # opti for multi-classes classification
  optimizer = optim.Adam(model.parameters(), lr=0.005)
  return criterion, optimizer

---

## 4.2 Training Loop

In [9]:
def training(model, dataloader, num_epochs=15):
  criterion, optimizer = init_training(model)
  model.train()

  print("Start of model training")
  for epoch in range(num_epochs):
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    for batch_X, batch_y in dataloader:
      optimizer.zero_grad()
      outputs = model(batch_X)
      loss = criterion(outputs, batch_y)
      loss.backward()
      optimizer.step()

      # loss
      total_loss += loss.item() * batch_X.size(0)

      # accuracy
      _, predicted = torch.max(outputs, 1)
      correct_predictions += (predicted == batch_y).sum().item()

      total_samples += batch_y.size(0)

    epoch_loss = total_loss / total_samples
    epoch_acc = correct_predictions / total_samples
    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc * 100:.2f}%")

  print("Complete")
  return model

---

# 5. modeles entrainer sur weight = 1

In [10]:
weight = 1

## 5.1 init model

In [11]:
possible_errors = generate_possible_errors(weight)
num_classes = len(possible_errors)

print("Number of possible errors:", num_classes)

Number of possible errors: 28


In [12]:
data = generate_torch_dataset(num_samples=500, weight=weight)

Generating sample 0/500
Generating sample 50/500
Generating sample 100/500
Generating sample 150/500
Generating sample 200/500
Generating sample 250/500
Generating sample 300/500
Generating sample 350/500
Generating sample 400/500
Generating sample 450/500
Dataset Size: torch.Size([500, 8])


In [13]:
model = SurfaceCodeDecoderMLP(input_dim=8, hidden_dim=128, output_dim=num_classes)

In [14]:
model = training(model, data, num_epochs=15)

Start of model training
Epoch [1/15] | Loss: 3.1345 | Accuracy: 26.40%
Epoch [2/15] | Loss: 2.2962 | Accuracy: 58.00%
Epoch [3/15] | Loss: 1.1663 | Accuracy: 78.00%
Epoch [4/15] | Loss: 0.5000 | Accuracy: 86.60%
Epoch [5/15] | Loss: 0.2779 | Accuracy: 86.40%
Epoch [6/15] | Loss: 0.2205 | Accuracy: 87.40%
Epoch [7/15] | Loss: 0.2029 | Accuracy: 87.80%
Epoch [8/15] | Loss: 0.2097 | Accuracy: 87.20%
Epoch [9/15] | Loss: 0.2206 | Accuracy: 85.80%
Epoch [10/15] | Loss: 0.2045 | Accuracy: 87.40%
Epoch [11/15] | Loss: 0.2052 | Accuracy: 85.60%
Epoch [12/15] | Loss: 0.1926 | Accuracy: 88.40%
Epoch [13/15] | Loss: 0.1998 | Accuracy: 88.80%
Epoch [14/15] | Loss: 0.2006 | Accuracy: 84.40%
Epoch [15/15] | Loss: 0.1942 | Accuracy: 88.00%
Complete


---

## 5.2 Test comparaison avec lookuptable (weight = 1)

---

## 5.3 teste avec des erreurs qui ont weight = 2

---

# 6. modeles entrainer sur weight = 2

In [15]:
weight = 2

## 6.1 init model

In [16]:
possible_errors = generate_possible_errors(weight)
num_classes = len(possible_errors)

print("Number of possible errors:", num_classes)

Number of possible errors: 352


In [17]:
data = generate_torch_dataset(num_samples=500, weight=weight)

Generating sample 0/500
Generating sample 50/500
Generating sample 100/500
Generating sample 150/500
Generating sample 200/500
Generating sample 250/500
Generating sample 300/500
Generating sample 350/500
Generating sample 400/500
Generating sample 450/500
Dataset Size: torch.Size([500, 8])


In [18]:
model = SurfaceCodeDecoderMLP(input_dim=8, hidden_dim=128, output_dim=num_classes)

In [19]:
model = training(model, data, num_epochs=15)

Start of model training
Epoch [1/15] | Loss: 5.8451 | Accuracy: 2.20%
Epoch [2/15] | Loss: 5.3291 | Accuracy: 7.20%
Epoch [3/15] | Loss: 4.5903 | Accuracy: 13.00%
Epoch [4/15] | Loss: 3.6465 | Accuracy: 26.20%
Epoch [5/15] | Loss: 2.5797 | Accuracy: 39.80%
Epoch [6/15] | Loss: 1.8137 | Accuracy: 44.40%
Epoch [7/15] | Loss: 1.4518 | Accuracy: 49.60%
Epoch [8/15] | Loss: 1.3341 | Accuracy: 47.00%
Epoch [9/15] | Loss: 1.2300 | Accuracy: 51.40%
Epoch [10/15] | Loss: 1.1989 | Accuracy: 48.40%
Epoch [11/15] | Loss: 1.1839 | Accuracy: 49.40%
Epoch [12/15] | Loss: 1.1204 | Accuracy: 52.60%
Epoch [13/15] | Loss: 1.1069 | Accuracy: 51.40%
Epoch [14/15] | Loss: 1.0776 | Accuracy: 49.40%
Epoch [15/15] | Loss: 1.0775 | Accuracy: 49.00%
Complete


---

## 6.2 comparaison avec lookuptable (weight = 1)


---

## 6.3 teste avec des erreurs qui ont weight = 2

---

# References

- **Machine Learning for Physics** (2025-2026)
  Midlands Physics Alliance Graduate School (MPAGS): Course Module, Access: [https://warwick.ac.uk/fac/sci/physics/mpags/modules/comp/machinelearning/25-26/](https://warwick.ac.uk/fac/sci/physics/mpags/modules/comp/machinelearning/25-26/)

- **PyTorch documentation** https://docs.pytorch.org/docs/main/
  